## Import Libraries

In [ ]:
!pip install qiskit

!pip install qiskit-aer

!pip install tdqr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 66.4 MB/s eta 0:00:00


In [1]:
import numpy as np
from qiskit import QuantumCircuit,transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import Statevector,DensityMatrix,state_fidelity,partial_trace, Operator
from matplotlib import pyplot as plt
from functools import reduce
from scipy.linalg import expm
import pandas as pd
from qiskit_aer import AerSimulator
import qiskit.circuit.classical as qiskit_classical
from IPython.display import display
#from qiskit.opflow import I, Z, StateFn, PauliExpectation, CircuitSampler
#from qiskit import Aer, execute, transpile
from qiskit_aer.primitives import SamplerV2 as AerSampler
# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)


Hamiltonian:

$H=-h(X_1+X_2)-J Z_1 Z_2$

Trotter formula:

$e^{-iH t}=(e^{i h X_1 \Delta t}e^{i h X_2 \Delta t}e^{i J Z_1 Z_2 \Delta t})^{N_{\text{Trot}}},$

where $\Delta t=t/N_{\text{Trot}}$.

## AER Pure SWAPNET

### Set SWAPNET

In [62]:
class build_ising_class:
    def __init__(self, d, steps, t, J, h):
        self.d = d
        self.steps = steps
        self.t = t
        self.J = J
        self.h = h

    def get_trotterized_ising_circuit(self):
        
        """
        Returns a QuantumCircuit implementing a trotterized Ising evolution for d qubits.

        H = - J * sum(Z_i Z_{i+1}) - h * sum(X_i)
        U = exp(-i H t) 
        """
        t = self.t
        steps = self.steps
        d = self.d
        J = self.J
        h = self.h

        dt = t / steps
        qc = QuantumCircuit(d)

        for _ in range(steps):
            # Apply ZZ interactions (Z_i Z_{i+1})
            for i in range(d - 1):
                qc.cx(i, i + 1)
                qc.rz(-2 * J * dt, i + 1)
                qc.cx(i, i + 1)

            # Apply transverse field X terms (X_i)
            for i in range(d):
                qc.rx(-2 * h * dt, i)

        return qc

    def apply_ising_to_registers(self, qc):
        """
        Apply trotterized Ising circuit to registers q1, q2, q3 in a 4d-sized register circuit.
        """
        d = self.d
        ising = self.get_trotterized_ising_circuit()

        # Convert to instruction and append to registers q1, q2, q3
        ising_inst = ising.to_instruction()
        for reg in [1, 2, 3]:
            qc.append(ising_inst, [reg * d + i for i in range(d)])

        return qc

    def get_trotterized_ising_statevector(self):
        """
        Returns the statevector from the trotterized Ising evolution of d qubits.
        """
        qc = self.get_trotterized_ising_circuit()
        qc.save_statevector()

        simulator = AerSimulator()
        result = simulator.run(transpile(qc, simulator)).result()
        return result.get_statevector()

def get_QPA_circuit(d, N, ising_class,test_depolarization, before_measure= False):
    #FUNCTION TO GET QPA CIRCUIT

    cr_q0 = ClassicalRegister(d,name='control')
    cr_q3 = ClassicalRegister(d,name='measure')
    qr_all = QuantumRegister(4*d)

    # Initialize quantum circuit with classical registers
    qcSWAP = QuantumCircuit(qr_all, cr_q0)  # Extra qubit and classical bit for parity check
    
    qc = ising_class.apply_ising_to_registers(qcSWAP) #Apply trotterized Ising circuit, 1)
    if test_depolarization:
       raise ValueError('Depolarization not implemented yet')
    def recursive(N,qc):
      for k in range(d):
        qc.reset(k)
      qc.h(0)#q0_firstqbit = |0>+|1>/sqrt2
      for k in range(d-1):
        qc.cx(0,1+k)#q0 = |0000...> + |1111...>/sqrt2
      # Apply the first CSWAP gate controlled by q0, targeting q1 and q2
      for k in range(d):
        qc.cswap(0+k, k+d, k+2*d)#|+>_k x SYM12_k + |->_k x AntiSYM12_k /norm

      # Apply the second Hadamard gate to q0
      for k in range(d):
        qc.h(k) #|0> x SYM12 + |1> x AntiSYM12 /norm
      # Measure qubits 0 to d-1 into classical bits 0 to d-1




      for i in range(d): #Measure the control registers and find z
          qc.measure(i, cr_q0[i])
          if i==0:
            parity_control = qiskit_classical.expr.lift(cr_q0[i])
          else:
            parity_control = qiskit_classical.expr.bit_xor(parity_control, cr_q0[i])

      with qc.if_test(parity_control) as _else:
        #--------Z=1
        pass
      with _else:
        #---------Z = 0
        # qc.x(d+1) # Good test to make sure it's working
        for k in range(d):
          qc.swap(k+2*d, k+3*d) #Swap q2 with q3
        if N!=1:
          qc = recursive(N-1,qc) #Do it again unless it was the final iteration of the SWAPNET
      return qc
    if N!=0:
      qc = recursive(N,qcSWAP)
    else:
      qc = qcSWAP
    # Gets Measure register q3 and save in the classical register
    if not before_measure:
      print('Measuring')
      qc.add_register(cr_q3)
      for i in range(d):
          # qc.z(3*d+i)
          qc.measure(3*d+i, cr_q3[i]) 
    return qc

def run_qc_and_return_state(qc):

    # Select Aer Simulator backend
    simulator = AerSimulator()

    def execute_circuit_on_state(qc):
        """ Executes a circuit on the AerSimulator and returns the state result. """
        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled).result()
        return result.get_statevector(qc_transpiled)

    qc.save_statevector()
    state = execute_circuit_on_state(qc)

    return state

def sample_qc_and_return_distribution(qc,shots=1024,sampler = AerSampler(), name='measure'):
    #RUN THE CIRCUIT AND MEASURE CLASSICALLY THE STATE IN REGISTER Q3, SHOULD RETURN A SEQUENCE OF BITS OF d DIMENSIONS FOR EACH SHOT, REPRESENTING THE FINAL STATE MEASURED
    pass_manager = generate_preset_pass_manager(3, AerSimulator())
    isa_circuit = pass_manager.run(qc)
    qc_transpiled = transpile(isa_circuit)
    result = sampler.run([qc_transpiled],shots=shots).result()
    pub_result = result[0]
    if name== 'measure':
      counts = pub_result.data.control.get_counts()
    if name == 'shadow':
      counts = pub_result.data.shadow.get_counts()
    if name =='measure':
      counts = pub_result.data.measure.get_counts()
    return counts



# Run the function
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h', 'x'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
t=0
J=1
h=1
steps = 2
d=4

ising_class = build_ising_class(d, steps, t, J, h)
epsilon = 0.0
N_qpa = 2
QPA = get_QPA_circuit(d, N_qpa, ising_class,test_depolarization=False)
sampler = AerSampler(options=dict(backend_options=dict(noise_model=noise_model)))
shots=1024

data = sample_qc_and_return_distribution(QPA,shots=shots,sampler=sampler)
print(data)

trotterized_state = ising_class.get_trotterized_ising_statevector()
display(trotterized_state.draw('latex'))
# psi_purified = run_qc_and_return_state(QPA)
#print(transpile(QPA).draw())
# display(psi_purified.draw('latex'))



Measuring
{'0000': 955, '0001': 14, '0010': 11, '0110': 1, '1000': 18, '0100': 23, '0011': 1, '1010': 1}


<IPython.core.display.Latex object>

### Run Shadow Tomography

In [118]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import HGate, SGate

import random
from collections import defaultdict
from qiskit.quantum_info import Pauli, SparsePauliOp


def generate_random_pauli_basis(d):
    """Generates a list of random Pauli bases ('X', 'Y', 'Z') for d qubits."""
    return [random.choice(['X', 'Y', 'Z']) for _ in range(d)]
    # return [random.choice(['X', 'Y', 'Z'])]*d

def apply_basis_rotation(circuit, basis, start):
    """Applies the inverse of basis rotation before measurement."""
    for i, b in enumerate(basis):
        if b == 'X':
            circuit.h(start+i)
        elif b == 'Y':
            circuit.sdg(start+i)
            circuit.h(start+i)
    return circuit

def get_shadowed_circuit(base_circuit, basis, start, pass_manager = generate_preset_pass_manager(3, AerSimulator())):
    """Returns a new circuit with basis rotations and measurements for shadow estimation."""
    circuit = base_circuit.copy()
    circuit = apply_basis_rotation(circuit, basis, start)
    cr = ClassicalRegister(len(basis), 'c_shadow')
    circuit.add_register(cr)
    for q in range(len(basis)):
        circuit.measure(q+start, cr[q])

    isa_circuit = pass_manager.run(circuit)
    qc_transpiled = transpile(isa_circuit)
    return qc_transpiled

def create_shadow_batch(base_circuit, d, start, num_randomizations):
    """Creates a list of circuits with randomized Pauli measurements."""
    shadow_circuits = []
    bases = []
    for _ in range(num_randomizations):
        basis = generate_random_pauli_basis(d)
        shadow_circuits.append(get_shadowed_circuit(base_circuit, basis, start))
        bases.append(basis)
    return shadow_circuits, bases

def run_sampler_on_circuits(sampler, circuits, shots=100):
    """Runs sampler and returns raw measurement outcomes (bitstrings)."""
    return sampler.run(circuits, shots=shots).result()




def single_qubit_inverse_channel(bit, basis):
    """Return the inverse map for a single-qubit measurement."""
    if basis == 'Z':
        projector = np.array([[1, 0], [0, 0]]) if bit == 0 else np.array([[0, 0], [0, 1]])
        U = np.eye(2)
    elif basis == 'X':
        U = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
        projector = np.array([[1, 0], [0, 0]]) if bit == 0 else np.array([[0, 0], [0, 1]])
    elif basis == 'Y':
        U = np.array([[1, -1j], [1, 1j]]) / np.sqrt(2)
        projector = np.array([[1, 0], [0, 0]]) if bit == 0 else np.array([[0, 0], [0, 1]])
    else:
        raise ValueError("Invalid basis")

    rotated = U @ projector @ U.conj().T
    shadow = 3 * rotated - np.eye(2)
    return shadow


def reconstruct_shadow(measured_bitstring, basis):
    """Reconstruct the classical shadow as a density matrix (not Pauli yet)."""
    shadow = 1
    for bit, b in zip(reversed(measured_bitstring), basis):
        rho = single_qubit_inverse_channel(int(bit), b)
        shadow = np.kron(shadow, rho)  # tensor product across qubits
    return shadow


def estimate_expectation_from_shadow(counts_basis_pairs, observable):
    """
    Optimized fidelity estimation using shadow tomography.
    Uses histogram data to avoid redundant reconstruction of shadow matrices.
    """
    if isinstance(observable, SparsePauliOp):
        obs_matrix = observable.to_matrix()
    else:
        obs_matrix = observable

    total_shadow = np.zeros_like(obs_matrix, dtype=complex)
    total_counts = 0

    for counts, basis in counts_basis_pairs:
        for bitstring, count in counts.items():
            rho = reconstruct_shadow(bitstring, basis)
            total_shadow += count * rho
            total_counts += count

    avg_shadow = total_shadow / total_counts
    avg_shadow /= np.trace(avg_shadow)
    return np.real(np.einsum('ij,ji->', obs_matrix, avg_shadow))


import time

def estimate_fidelity_with_shadow(base_QPA, base_psi, d, shots, num_randomizations, sampler_QPA, start):
    start_total = time.time()

    # Step 1: Create shadow circuits
    t0 = time.time()
    QPA_circuits, bases_QPA = create_shadow_batch(base_QPA, d, start, num_randomizations)
    print(f"[Timing] Created {num_randomizations} shadow circuits in {time.time() - t0:.2f} seconds.")

    # Step 2: Run sampler
    t1 = time.time()
    QPA_results = run_sampler_on_circuits(sampler_QPA, QPA_circuits, shots)
    print(f"[Timing] Ran {len(QPA_circuits)} circuits with {shots} shots each in {time.time() - t1:.2f} seconds.")

    # Step 3: Extract counts
    t2 = time.time()
    counts_basis_pairs = []
    for res, basis in zip(QPA_results, bases_QPA):
        counts = res.data.c_shadow.get_counts()
        counts_basis_pairs.append((counts, basis))
    print(f"[Timing] Processed sampler results into histograms in {time.time() - t2:.2f} seconds.")

    # Step 4: Compute projector
    t3 = time.time()
    psi = Statevector.from_instruction(base_psi).data
    psi_projector = np.outer(psi, psi.conj())
    print(f"[Timing] Constructed projector from statevector in {time.time() - t3:.2f} seconds.")

    # Step 5: Estimate fidelity
    t4 = time.time()
    fidelity = estimate_expectation_from_shadow(counts_basis_pairs, psi_projector)
    print(f"[Timing] Estimated expectation from shadows in {time.time() - t4:.2f} seconds.")

    print(f"[Timing] Total time for estimate_fidelity_with_shadow: {time.time() - start_total:.2f} seconds.\n")

    return fidelity


In [123]:
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h', 'x'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
t=0
J=1
h=1
steps = 2
d=2

ising_class = build_ising_class(d, steps, t, J, h)
ising_circuit = ising_class.get_trotterized_ising_circuit()

N_qpa = 2
QPA_circuit = get_QPA_circuit(d, N_qpa, ising_class,test_depolarization=False,before_measure=True)

noisy_sampler = AerSampler(options=dict(backend_options=dict(noise_model=noise_model)))
pure_sampler = AerSampler()

shots=1024*10


fidelity_shadow = estimate_fidelity_with_shadow(QPA_circuit, ising_circuit, d, shots, num_randomizations=200, sampler_QPA=noisy_sampler, start=3*d)
#fidelity_shadow = estimate_fidelity_with_shadow(ising_circuit, ising_circuit, d, shots, num_randomizations=200, sampler_QPA=noisy_sampler, start=0)
print('Fidelity Obtained with shadow tomography:', fidelity_shadow)


QPA = get_QPA_circuit(d, N_qpa, ising_class,test_depolarization=False)
data = sample_qc_and_return_distribution(QPA,shots=shots,sampler=noisy_sampler)
fidelity_test = data['0'*d]/shots
print('Fidelity Obtained with Counting states:', fidelity_test)




[Timing] Created 200 shadow circuits in 3.34 seconds.
[Timing] Ran 200 circuits with 10240 shots each in 310.96 seconds.
[Timing] Processed sampler results into histograms in 1.14 seconds.
[Timing] Constructed projector from statevector in 0.00 seconds.
[Timing] Estimated expectation from shadows in 0.03 seconds.
[Timing] Total time for estimate_fidelity_with_shadow: 315.47 seconds.

Fidelity Obtained with shadow tomography: 0.8329975585937502
Measuring
Fidelity Obtained with Counting states: 0.96123046875


In [7]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import HGate, SGate

import random
from collections import defaultdict

def apply_random_pauli_basis(qc, basis_list,start=0):
    """Apply a basis-changing gate to each qubit in basis_list (either 'X', 'Y', or 'Z')"""
    for i, basis in enumerate(basis_list):
        if basis == 'X':
            qc.h(start+i)
        elif basis == 'Y':
            qc.sdg(start+i)
            qc.h(start+i)
        # For Z basis, do nothing (computational basis)
    return qc

def generate_random_pauli_basis(d):
    """Generate a random basis list of length d ('X', 'Y', 'Z')"""
    return [random.choice(['X', 'Y', 'Z']) for _ in range(d)]
    # return [random.choice(['A', 'B', 'C']) for _ in range(d)]
def create_get_statistics(sampler):
    def get_statistics(qc, shots):
        """Get statistics for a quantum circuit"""
        statistics = sample_qc_and_return_distribution(qc,shots=shots,sampler=sampler,name='shadow')
        return statistics
    return get_statistics   

def measure_fidelity_shadow(QPA_circuit, psi_circuit, get_statistics_QPA, get_statistics_psi,d, shots=1024, num_randomizations=3):
    """Estimate fidelity using randomized measurements (shadow tomography)"""
    fidelity_estimates = []
    fidelity_tests=[]
    for _ in range(num_randomizations):
        # Step 1: Random basis for each qubit
        basis = generate_random_pauli_basis(d)
        print('Basis:', basis)
        # Step 2: Create modified circuits
        qc_QPA = QPA_circuit.copy()
        qc_QPA = apply_random_pauli_basis(qc_QPA, basis,start=3*d)

        qc_psi = psi_circuit.copy()
        qc_psi = apply_random_pauli_basis(qc_psi, basis)

        # Step 3: Measure all qubits
        crQPA = ClassicalRegister(d, name='shadow')
        crpsi = ClassicalRegister(d, name='shadow')
        qc_QPA.add_register(crQPA)
        qc_psi.add_register(crpsi)

        for i in range(d):
            qc_QPA.measure(3*d + i, crQPA[i])  # Only measure q3
        for i in range(d):
            qc_psi.measure(i, crpsi[i])  # Measure all
        # Step 4: Get statistics for both
        stats_QPA = get_statistics_QPA(qc_QPA, shots)
        stats_psi = get_statistics_psi(qc_psi, shots)

        

        # Step 5: Compute overlap of distributions (classical fidelity)
        keys = set(stats_QPA) & set(stats_psi)
        fidelity_snapshot = sum(np.sqrt(stats_QPA[k]/shots * stats_psi[k]/shots) for k in keys) #sum(stats_QPA[k] * stats_psi[k] for k in keys)
        print('Stats:', stats_QPA, stats_psi)
        print('Fidelity:', fidelity_snapshot)
        fidelity_estimates.append(fidelity_snapshot)
    return np.mean(fidelity_estimates), np.std(fidelity_estimates)


In [ ]:
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h', 'x'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
t=0
J=1
h=1
steps = 2
d=2

ising_class = build_ising_class(d, steps, t, J, h)
ising_circuit = ising_class.get_trotterized_ising_circuit()

N_qpa = 2
QPA_circuit = get_QPA_circuit(d, N_qpa, ising_class,test_depolarization=False,before_measure=True)

noisy_sampler = AerSampler(options=dict(backend_options=dict(noise_model=noise_model)))
pure_sampler = AerSampler()

shots=1024*5


get_statistics_QPA = create_get_statistics(sampler)
get_statistics_psi = create_get_statistics(pure_sampler)

fidelity_shadow, _ = measure_fidelity_shadow(QPA_circuit, ising_circuit, get_statistics_QPA, get_statistics_psi,d, shots=1024, num_randomizations=5)
print('Fidelity Obtained with shadow tomography:', fidelity_shadow)


QPA = get_QPA_circuit(d, N_qpa, ising_class,test_depolarization=False)
data = sample_qc_and_return_distribution(QPA,shots=shots,sampler=noisy_sampler)
fidelity_test = data['0'*d]/shots
print('Fidelity Obtained with Counting states:', fidelity_test)




Basis: ['Z', 'Y']
Stats: {'00': 493, '10': 515, '11': 8, '01': 8} {'00': 493, '10': 531}
Fidelity: 0.9921277450704137
Basis: ['Y', 'X']
Stats: {'00': 260, '11': 241, '01': 259, '10': 264} {'11': 228, '01': 282, '00': 254, '10': 260}
Fidelity: 0.9996486211980335
Basis: ['Y', 'Z']
Stats: {'00': 474, '01': 534, '10': 6, '11': 10} {'00': 504, '01': 520}
Fidelity: 0.991917375870794
Basis: ['Z', 'Y']
Stats: {'10': 528, '00': 482, '01': 9, '11': 5} {'10': 499, '00': 525}
Fidelity: 0.9925156157205174
Basis: ['Y', 'X']
Stats: {'11': 279, '10': 256, '00': 265, '01': 224} {'00': 256, '10': 255, '01': 264, '11': 249}
Fidelity: 0.9987432674931045
Fidelity Obtained with shadow tomography: 0.9949905250705726
before measure
Fidelity Obtained with Counting states: 0.9611328125


## DO THE PLOTTING - CURRENTLY HAS NO METHOD TO FIND FIDELITY

In [14]:
#DO THE PLOTTING -------------------------
import numpy as np
import matplotlib.pyplot as plt
import time  # Import time module for tracking execution time
from tqdm import tqdm
list_of_lambda = [i * 0.05 for i in range(20)]
list_of_Nqpa = [0, 1, 2]
purified_fidelity = {i: [] for i in list_of_Nqpa}
theorical_fidelity=[]
d = 2
shots = 1024
debug = False # Set to False to disable print statements

for LAMBDA in tqdm(list_of_lambda, desc="Lambda Loop"):
    # Theoretical max fidelity
    if d==2:#1/8 (-2 + \[Lambda]) (1 + \[Lambda]) (-4 + 3 \[Lambda])
        theorical_fidelity.append(1/8 * (-2 + LAMBDA) * (1 + LAMBDA) * (-4 + 3*LAMBDA))
    elif d==3: #1/96 (-8 + 7 \[Lambda]) (-12 + 7 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/96 * (-8 + 7*LAMBDA) * (-12 + 7*(-1 + LAMBDA)*LAMBDA))
    elif d==4:#1/128 (-16 + 15 \[Lambda]) (-8 + 5 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/128 * (-16 + 15*LAMBDA) * (-8 + 5*(-1 + LAMBDA)*LAMBDA))
    else:#Just give 0 for now
        theorical_fidelity.append(0)
        

    if debug:
        print(f'Running: Epsilon={LAMBDA}')
        
    for Nqpa in list_of_Nqpa:
        if debug:
            print(f'Running: Epsilon={LAMBDA}, Nqpa={Nqpa}')
        start_time = time.time()

        QPA = get_QPA_circuit(d, LAMBDA, Nqpa)
        mid_time_qpa = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the QPA in {mid_time_qpa - start_time:.2f} seconds')

        results = run_qc_and_return_result(QPA, shots)
        mid_time_results = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Results in {mid_time_results - mid_time_qpa:.2f} seconds')

        memory = results.get_memory()
        fidelity = memory.count('0' * d) / len(memory)
        purified_fidelity[Nqpa].append(fidelity)

        end_time = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Fidelity in {end_time - mid_time_results:.2f} seconds')
            print(f'Total time for Epsilon={LAMBDA}, Nqpa={Nqpa}: {end_time - start_time:.2f} seconds\n')


# Plot results
for Nqpa in list_of_Nqpa:
    fidelity = np.array(purified_fidelity[Nqpa])
    if Nqpa==0:
        label = 'Base Fidelity'
    else:
        label = f'Fidelity after {Nqpa} Swapnets'
    plt.errorbar(list_of_lambda, purified_fidelity[Nqpa],
                 yerr=np.sqrt(fidelity/shots), label=label)

plt.plot(list_of_lambda, theorical_fidelity, label='Theoretical Maximum Fidelity')

plt.xlabel('Lambda')
plt.ylabel('Fidelity')
plt.title('Fidelity vs Lambda for Different Nqpa')
plt.legend()
plt.show()


Lambda Loop:   0%|          | 0/20 [00:00<?, ?it/s]

NameError: name 'run_qc_and_return_result' is not defined